In [26]:

from langchain_core.documents import Document

text = """
LangChain 是一个用于构建 LLM 应用的框架，常用于开发 AI Agent、RAG（Retrieval-Augmented Generation）系统以及工作流编排。

很多开发者会把它当成“大模型时代的 Spring Framework”，因为它提供了 prompt management、memory、tool calling、retriever、agent orchestration 等模块化能力。

在检索增强生成场景中，开发者通常会将文本 chunk 存储到 vector database，例如 Chroma、FAISS、Milvus、Pinecone 或 Weaviate。

向量数据库（Vector Store）本质上保存的是 embedding 向量，而不是原始文本本身。用户查询时，系统会把 query 转换成 embedding，然后进行 semantic similarity search。

有些系统使用 cosine similarity，也有些使用 ANN（Approximate Nearest Neighbor）算法，比如 HNSW 或 IVF。

MultiQueryRetriever 是 LangChain 中一种提升召回率（recall）的技术。它会让 LLM 自动生成多个不同表达的查询语句，例如：

- “如何提高 RAG 检索效果”
- “怎样优化知识库召回”
- “improve retrieval quality in vector search”
- “增强 embedding 检索命中率的方法”

这些 query 会分别执行向量检索，然后对结果进行 merge 和 deduplicate。

这种方法特别适用于：
1. 用户问题表达模糊
2. chunk 中关键词不一致
3. 存在大量同义词
4. embedding 模型语义能力有限

例如，一篇文档中写的是“高并发缓存系统”，但用户问的是“redis scalability architecture”，单一 query 可能无法命中。

同样，“authentication” 和 “登录鉴权” 在某些 embedding 模型中距离并不近。

因此，多查询（multi query）能够从不同角度改写用户问题，提高 retrieval recall。

除了 MultiQueryRetriever，还有一些常见 RAG 优化技术：

- HyDE（Hypothetical Document Embeddings）
- Parent Document Retriever
- Contextual Compression Retriever
- Reranker
- BM25 + Vector Hybrid Search
- Self Query Retriever

在生产环境中，很多团队会结合 rerank model 与 hybrid retrieval 一起使用，以降低 hallucination 并提升 answer grounding。

LangGraph 是 LangChain 生态中的一个状态机工作流框架，用于构建 multi-agent systems、long-running workflow 和复杂推理链。

有些开发者认为：
- LangChain 更像应用层抽象
- LangGraph 更偏 orchestration engine
- DeepAgents 更像高级 agent runtime

在 AI Coding 场景中，RAG 常被用于：
- 代码库问答
- 文档检索
- API 搜索
- 企业知识库
- Wiki assistant

一个典型 pipeline 通常包括：

Document Loader → Text Splitter → Embedding Model → Vector Store → Retriever → LLM → Response Generator

如果 chunk size 太小，可能导致 context fragmentation；
如果 chunk 太大，又可能降低 embedding precision。

因此，chunk overlap、chunking strategy、metadata filtering 都会影响最终 retrieval performance。"""

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)
chunks = splitter.split_text(text)
print(len(chunks))

6


In [31]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama

embedding_model = OllamaEmbeddings(
    model="qwen3-embedding:8b",
    base_url="http://192.168.31.198:11434",
)
vector_store = Chroma(
    collection_name="c3-5",  # 给向量库起一个名字
    embedding_function=embedding_model,
    persist_directory="./chroma_db" # 指定数据存放的文件夹
)
# 删除同名 collection，并重新创建一个空 collection
vector_store.reset_collection()
vector_store.add_texts(chunks)

['975f3542-bf09-4ed1-a437-0920341919a8',
 '7bce5678-0885-4517-8ca2-00f3536898c3',
 'c82d4ab6-e449-464b-97fe-447909958494',
 '20f17471-fb66-4965-9509-3c251bc35730',
 'a3d6ffee-540c-46bf-978c-582e10dde18d',
 'e5deef5a-f051-4bc6-8de3-fa2a78b3a990']

## 迭代式回答：

原始问题 -> LLM -> 三个问题 Q1, Q2, Q3

拆解：

Q1 -> 检索  ->  LLM = 答案 1

Q2 -> 检索 + 答案1 -> LLM = 答案 2

Q3 -> 检索 + 答案 2 -> LLM = 最终答案

In [62]:
from typing import List
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
import os
load_dotenv()
model = ChatOllama(
    base_url="http://192.168.31.198:11434",
    model = "qwen3:8b",
    temperature=0
)
class Questions(BaseModel):
    questions: List[str] = Field(description="语义相近的问题列表")

structured_llm = model.with_structured_output(
    Questions,
    method="json_mode"
)
query = """
LangChain、LangGraph 和 Deep Agents 的定位分别是什么？
如果我要开发一个带 RAG、长期状态和复杂工作流的 AI 应用，
应该如何组合使用它们？"""
prompt = ChatPromptTemplate.from_template(
    "你是一个乐于助人的 AI 助手，可以针对一个输入问题，生成多个相关的子问题。\n"
    "目标是将输入的问题分解成一组可以独立回答的子问题或者子任务\n"
    "生成与以下问题相关的多个搜索查询。\n"
    """原问题：{query}
    请仅返回 JSON 对象，输出三个问题/子查询，格式如下：
    {{"questions": ["问题1", "问题2", "问题3"]}}"""
)
sub_query_chain =  {"query": RunnablePassthrough() } | prompt | structured_llm
questions = sub_query_chain.invoke(query)

In [63]:
for question in questions.questions:
    print(question)

LangChain、LangGraph 和 Deep Agents 的核心定位与适用场景分别是什么？
如何将 RAG 功能与 LangChain、LangGraph 结合实现复杂工作流？
在需要长期状态管理的 AI 应用中，如何整合 LangChain、LangGraph 和 Deep Agents？


In [64]:
# 迭代查询三个问题，把上一个问题的答案作为下一个问题的问题
decomposition_prompt = ChatPromptTemplate.from_template(
"""
这是你要回答的问题：
{query}

------
这是所有可用的背景问题和答案对:
{qa_pairs}

------
这是与问题相关的额外背景信息:
{retrieve_data}
"""
)
#retriever = vector_store.as_retriever(search_type="mmr")
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
def print_prompt(prompt) -> str:
    print("="*50)
    print(prompt)
    return prompt
chain =  decomposition_prompt | print_prompt | model | StrOutputParser()

In [65]:
qa_pairs = ""
result = ""
def combin_document(docs: List[Document]) -> str:
    doc_str = ""
    for doc in docs:
        doc_str += f"{doc.page_content}\n"
    return doc_str

for index, question in enumerate(questions.questions, start=1):
    retrieve_data = combin_document(retriever.invoke(question))
    result = chain.invoke(
        {"query": question,
         "qa_pairs": qa_pairs,
         "retrieve_data": retrieve_data}
    )
    qa_pairs += f"{index}. {question}  回答:{result}\n"

print(result)


messages=[HumanMessage(content='\n这是你要回答的问题：\nLangChain、LangGraph 和 Deep Agents 的核心定位与适用场景分别是什么？\n\n------\n这是所有可用的背景问题和答案对:\n\n\n------\n这是与问题相关的额外背景信息:\n有些开发者认为：\n- LangChain 更像应用层抽象\n- LangGraph 更偏 orchestration engine\n- DeepAgents 更像高级 agent runtime\n\n在 AI Coding 场景中，RAG 常被用于：\n- 代码库问答\n- 文档检索\n- API 搜索\n- 企业知识库\n- Wiki assistant\n\n一个典型 pipeline 通常包括：\n\nDocument Loader → Text Splitter → Embedding Model → Vector Store → Retriever → LLM → Response Generator\n\n如果 chunk size 太小，可能导致 context fragmentation；\n如果 chunk 太大，又可能降低 embedding precision。\nLangChain 是一个用于构建 LLM 应用的框架，常用于开发 AI Agent、RAG（Retrieval-Augmented Generation）系统以及工作流编排。\n\n很多开发者会把它当成“大模型时代的 Spring Framework”，因为它提供了 prompt management、memory、tool calling、retriever、agent orchestration 等模块化能力。\n\n在检索增强生成场景中，开发者通常会将文本 chunk 存储到 vector database，例如 Chroma、FAISS、Milvus、Pinecone 或 Weaviate。\n\n', additional_kwargs={}, response_metadata={})]
messages=[HumanMessage(content='\n这是你要回答的问题：\n如何将 RAG 功能与 LangChain、LangGraph 结合实现复杂工作流

In [66]:
print(qa_pairs)

1. LangChain、LangGraph 和 Deep Agents 的核心定位与适用场景分别是什么？  回答:LangChain、LangGraph 和 DeepAgents 的核心定位与适用场景如下：

---

### **1. LangChain**
**核心定位**  
LangChain 是一个 **LLM 应用开发框架**，专注于提供模块化工具和组件，支持构建 AI Agent、RAG（检索增强生成）系统以及复杂的工作流编排。它被开发者类比为“大模型时代的 Spring Framework”，强调灵活性和可扩展性。

**适用场景**  
- **RAG 系统开发**：通过集成检索器（Retriever）、向量数据库（如 Chroma、FAISS）和 LLM，实现文档检索与生成的闭环。  
- **AI Agent 构建**：支持工具调用、记忆管理、提示工程（Prompt Management）等模块化能力，适合需要多步骤交互的代理开发。  
- **工作流编排**：通过链式结构（Chain）和代理（Agent）管理复杂流程，例如从文档加载到最终响应生成的完整 pipeline。  

**典型技术栈**  
- 文档加载 → 分割 → 嵌入 → 向量存储 → 检索 → LLM → 响应生成（RAG 流程）。  

---

### **2. LangGraph**
**核心定位**  
LangGraph 是一个 **流程编排引擎（Orchestration Engine）**，专注于协调复杂的工作流和组件交互，强调流程的可组合性和动态性。它更偏向于“流程引擎”，而非直接提供 LLM 的功能。

**适用场景**  
- **多步骤 AI 任务**：例如需要串联多个 LLM 模块、外部 API 或数据库查询的复杂流程。  
- **动态流程管理**：支持条件分支、循环、状态转移等高级流程控制逻辑，适合需要灵活调整执行路径的场景。  
- **系统集成**：将不同组件（如 LangChain 的模块、外部工具）无缝连接，形成统一的执行流程。  

**典型技术栈**  
- 通过图结构定义流程节点（如检索、LLM 调用、数据库查询），并动态协调各节点的输入输出。  

---

### **3. DeepAgents**
**核心定位**  
De

In [67]:
print(result)

在需要长期状态管理的 AI 应用中，整合 **LangChain**、**LangGraph** 和 **DeepAgents** 需要结合三者的定位与能力，形成一个协同工作的系统。以下是整合策略、技术栈和具体实现步骤：

---

### **1. 核心角色与整合逻辑**
| **工具**         | **核心角色**                          | **整合逻辑**                                                                 |
|------------------|---------------------------------------|-----------------------------------------------------------------------------|
| **DeepAgents**   | 管理代理的长期状态、生命周期和决策逻辑 | 负责维护代理的长期记忆（如用户历史、任务进度）、状态持久化和智能决策。       |
| **LangGraph**    | 协调复杂流程与动态状态转移            | 定义代理的工作流（如用户请求 → 检索 → 生成 → 状态更新），支持条件分支和状态机。 |
| **LangChain**    | 提供模块化组件（记忆、工具、RAG）     | 实现具体功能模块（如记忆存储、工具调用、RAG 检索），作为 LangGraph 的节点。   |

---

### **2. 技术栈整合方案**
#### **(1) 状态管理与记忆**
- **DeepAgents**：  
  - 使用 `Memory` 模块（如 `langchain.memory`）存储长期上下文（如用户历史对话、任务状态）。  
  - 集成 `State` 管理，支持持久化（如数据库存储）和恢复（如从文件加载）。  
- **LangGraph**：  
  - 通过 `StateGraph` 定义状态转移逻辑，例如：  
    ```python
    from langgraph import StateGraph
    graph = StateGraph({"memory":